<a href="https://colab.research.google.com/github/ShadenAhmed/DataScience-Project/blob/main/DateScience_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#1. Collecting Data
###Phase 1

##1.1 Gold prices

In [ ]:
import yfinance as yf
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [ ]:

gold_symbol = "GC=F"
start_date = "2020-01-01"
end_date = "2026-1-31"

gold_raw_data = yf.download(gold_symbol, start=start_date, end=end_date)

print(gold_raw_data.head()) # first five days of 2020
print(gold_raw_data.tail()) # last days in 2026

gold_raw_data.to_csv("gold_prices_raw.csv")


/tmp/ipykernel_331/4200776802.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  gold_raw_data = yf.download(gold_symbol, start=start_date, end=end_date)
[*********************100%***********************]  1 of 1 completed

Price             Close         High          Low         Open Volume
Ticker             GC=F         GC=F         GC=F         GC=F   GC=F
Date                                                                 
2020-01-02  1524.500000  1528.699951  1518.000000  1518.099976    214
2020-01-03  1549.199951  1552.699951  1530.099976  1530.099976    107
2020-01-06  1566.199951  1580.000000  1560.400024  1580.000000    416
2020-01-07  1571.800049  1576.300049  1558.300049  1558.300049     47
2020-01-08  1557.400024  1604.199951  1552.300049  1579.699951    236
Price             Close         High          Low         Open  Volume
Ticker             GC=F         GC=F         GC=F         GC=F    GC=F
Date                                                                  
2026-01-26  5079.700195  5095.600098  5052.200195  5081.500000     180
2026-01-27  5079.899902  5079.899902  5079.899902  5079.899902      34
2026-01-28  5301.600098  5301.600098  5301.600098  5301.600098  112054
2026-01-29  53

## 1.2 Geopolitical News
Collected in phase 1 but the size of data were too small so in phase 2 we decide to collect more data from the same source to improve data quality for conducting analysis

In [14]:
!pip install gnews

from gnews import GNews
import pandas as pd
import time

google_news = GNews(
    language='en',
    max_results=100,
    start_date=(2020,1,1),
    end_date=(2026,1,1)
)

keywords = [
"gold geopolitical tensions",
"gold war",
"gold geopolitical risk",
"gold global conflict",
"gold russia ukraine",
"gold middle east tensions",
"gold political instability",
"gold economic sanctions",
"gold financial crisis",
"gold inflation crisis"
]

articles = []

for word in keywords:
    news = google_news.get_news(word)

    for n in news:
        articles.append({
            "title": n["title"],
            "date": n["published date"],
            "url": n["url"],
            "topic": word
        })

    time.sleep(2)

df = pd.DataFrame(articles)

print("Total geopolitical news:", len(df))

df.to_csv("raw_news.csv", index=False)

Total geopolitical news: 874


## 1.3 Global geopolitical conflict

Our secondary dataset

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd

df = pd.read_csv("geopolitical_conflict_risk_dataset.csv")

columns_needed = [
    "country",
    "month",
    "inflation_rate",
    "gdp_growth_pct",
    "border_disputes_count",
    "sanctions_active",
    "instability_score",
    "conflict_escalation_6m"
]

df = df[columns_needed]

df["month"] = pd.to_datetime(df["month"], errors="coerce")

df = df.dropna(subset=["month"])

df.head()

#2. Data Processing and Cleaning
###Phase 2


## 2.1 Geopolitical news data

In [ ]:
#imports + read

import pandas as pd
import re
import matplotlib.pyplot as plt

df = pd.read_csv("raw_news.csv")

#inspect dataset

df.head()
df.info()
df.isnull().sum()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 863 entries, 0 to 862
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   title   863 non-null    object
 1   date    863 non-null    object
 2   url     863 non-null    object
 3   topic   863 non-null    object
dtypes: object(4)
memory usage: 27.1+ KB


,title,date,url,topic
count,863,863,863,863
unique,771,434,781,10
top,Gold and the war in Sudan - Chatham House,"Mon, 22 Dec 2025 08:00:00 GMT",https://news.google.com/rss/articles/CBMiZkFVX...,gold geopolitical tensions
freq,5,19,4,100


**Inspection result:** date is stored as object type and requires conversion to datetime. Duplicate rows were found while no missing values were detected in the dataset.

In [ ]:
#cleaning the data

# date conversion to datetime
df["date"] = pd.to_datetime(
    df["date"],
    format="%a, %d %b %Y %H:%M:%S %Z",
    errors="coerce"
)


#Remove duplicates by url to be more accurate
df = df.drop_duplicates(subset=["url"])

#Even though no missing titles were found during the previous step,
#we still need to ensure the dataset is clean and reliable before further analysis.
df = df.dropna(subset=["title"])

#cleening the text
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

#side by side comparison
df["title_clean"] = df["title"].apply(clean_text)
df[["title", "title_clean"]].head(5)

df.head()


,title,date,url,topic,title_clean
0,Gold shines amid geopolitical uncertainties - ...,2024-05-15 07:00:00+00:00,https://news.google.com/rss/articles/CBMiiwFBV...,gold geopolitical tensions,gold shines amid geopolitical uncertainties wo...
1,Gold Rises to Record High on Rate-Cut Bets and...,2025-12-22 08:00:00+00:00,https://news.google.com/rss/articles/CBMitgFBV...,gold geopolitical tensions,gold rises to record high on ratecut bets and ...
2,"Gold gains as geopolitical risks, trade tensio...",2025-10-23 07:00:00+00:00,https://news.google.com/rss/articles/CBMinAFBV...,gold geopolitical tensions,gold gains as geopolitical risks trade tension...
3,Gold climbs to over 3-week high as dollar weak...,2025-06-02 07:00:00+00:00,https://news.google.com/rss/articles/CBMipgFBV...,gold geopolitical tensions,gold climbs to over 3week high as dollar weake...
4,Geopolitics affirms gold as a key strategic as...,2025-10-29 07:00:00+00:00,https://news.google.com/rss/articles/CBMid0FVX...,gold geopolitical tensions,geopolitics affirms gold as a key strategic as...


We will create a binary **event indicator** using specific keywords to identify geopolitical headlines, converting the news text into structured data that can be compared with gold price movements.

In [ ]:
#fearure engineering:
#convert unstucutured text to stucutured text by creating an event indicator

keywords = ["war", "conflict", "invasion", "attack",
            "sanction", "military", "tension", "crisis","crises"]

def detect_event(text):
    return 1 if any(word in text for word in keywords) else 0
df["event_indicator"] = df["title_clean"].apply(detect_event)
df["event_indicator"].value_counts()

#a monthgly event count
df["year_month"] = df["date"].dt.to_period("M")
monthly_events = df.groupby("year_month")["event_indicator"].sum()

monthly_events

/tmp/ipykernel_331/1015682664.py:13: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df["year_month"] = df["date"].dt.to_period("M")


,event_indicator
year_month,
2020-01,4
2020-02,0
2020-03,1
2020-04,2
2020-07,6
...,...
2025-08,14
2025-09,16
2025-10,16


News events will be organized by **month** to identify periods of higher geopolitical activity:

In [ ]:
#organize event timelines

#sort by date
df = df.sort_values("date")
df[["date", "title_clean", "event_indicator"]].head()

#draw barplot
#monthly_events.plot(kind="bar", figsize=(10,5), color="red")
#plt.title("Monthly Geopolitical Event Count")
#plt.xlabel("Year-Month")
#plt.ylabel("Number of Events")
#plt.xticks(rotation=90)
#plt.tight_layout()
#plt.show()

#high activity months - event clusters
#monthly_events.sort_values(ascending=False).head(5)

,date,title_clean,event_indicator
487,2020-01-06 08:00:00+00:00,gold powers towards 7year peak as investors ru...,1
484,2020-01-06 08:00:00+00:00,gold soars as middle east tensions brew perfec...,1
564,2020-01-07 08:00:00+00:00,gold price surges amid political uncertainty d...,0
510,2020-01-10 08:00:00+00:00,the week in mining gold rallies amid tensions ...,1
131,2020-01-11 08:00:00+00:00,going for gold for the rosies us department of...,1


 A significant spike was observed in **June 2025**, indicating a strong event cluster that can be compared with gold price behavior.

In [ ]:
#save clean dataset

clean_news_data = df[[
    "date",
    "title_clean",
    "event_indicator",
    "year_month"
]].copy()

clean_news_data.to_csv("clean_news_data.csv", index=False)

print("Clean dataset saved:", clean_news_data.shape)

Clean dataset saved: (781, 4)


##2.2 Gold prices data


In [ ]:
import pandas as pd

df = pd.read_csv("gold_prices_raw.csv")
#Initial Inspection
df.head()
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1532 entries, 0 to 1531
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Price   1532 non-null   object
 1   Close   1531 non-null   object
 2   High    1531 non-null   object
 3   Low     1531 non-null   object
 4   Open    1531 non-null   object
 5   Volume  1531 non-null   object
dtypes: object(6)
memory usage: 71.9+ KB


,Price,Close,High,Low,Open,Volume
count,1532,1531,1531,1531,1531,1531
unique,1532,1405,1386,1388,1395,883
top,2026-01-30,1815.9000244140625,1813.5,1784.0,1794.300048828125,6
freq,1,4,4,4,4,10


In [ ]:
#Renaming Columns
df.rename(columns={'Price': 'Date'}, inplace=True)
df.head()

,Date,Close,High,Low,Open,Volume
0,Ticker,GC=F,GC=F,GC=F,GC=F,GC=F
1,Date,NaN,NaN,NaN,NaN,NaN
2,2020-01-02,1524.5,1528.699951171875,1518.0,1518.0999755859375,214
3,2020-01-03,1549.199951171875,1552.699951171875,1530.0999755859375,1530.0999755859375,107
4,2020-01-06,1566.199951171875,1580.0,1560.4000244140625,1580.0,416


In [ ]:
#Removing Metadata Rows
df = df.iloc[2:].reset_index(drop=True)
df.head()

,Date,Close,High,Low,Open,Volume
0,2020-01-02,1524.5,1528.699951171875,1518.0,1518.0999755859375,214
1,2020-01-03,1549.199951171875,1552.699951171875,1530.0999755859375,1530.0999755859375,107
2,2020-01-06,1566.199951171875,1580.0,1560.4000244140625,1580.0,416
3,2020-01-07,1571.800048828125,1576.300048828125,1558.300048828125,1558.300048828125,47
4,2020-01-08,1557.4000244140625,1604.199951171875,1552.300048828125,1579.699951171875,236


In [ ]:
df['Date'] = pd.to_datetime(df['Date'])
#Converting Data Types
numeric_cols = ['Close','High','Low','Open','Volume']
df[numeric_cols] = df[numeric_cols].astype(float)
df.head()

,Date,Close,High,Low,Open,Volume
0,2020-01-02,1524.500000,1528.699951,1518.000000,1518.099976,214.0
1,2020-01-03,1549.199951,1552.699951,1530.099976,1530.099976,107.0
2,2020-01-06,1566.199951,1580.000000,1560.400024,1580.000000,416.0
3,2020-01-07,1571.800049,1576.300049,1558.300049,1558.300049,47.0
4,2020-01-08,1557.400024,1604.199951,1552.300049,1579.699951,236.0


In [ ]:
#Removing Duplicates
df.duplicated().sum()
df.drop_duplicates(inplace=True)

In [ ]:
#Sorting by Date
df.sort_values('Date', inplace=True)
df.reset_index(drop=True, inplace=True)

In [ ]:
#Calculate daily returns
df['Daily_Return'] = df['Close'].pct_change()
df.head()

,Date,Close,High,Low,Open,Volume,Daily_Return
0,2020-01-02,1524.500000,1528.699951,1518.000000,1518.099976,214.0,NaN
1,2020-01-03,1549.199951,1552.699951,1530.099976,1530.099976,107.0,0.016202
2,2020-01-06,1566.199951,1580.000000,1560.400024,1580.000000,416.0,0.010973
3,2020-01-07,1571.800049,1576.300049,1558.300049,1558.300049,47.0,0.003576
4,2020-01-08,1557.400024,1604.199951,1552.300049,1579.699951,236.0,-0.009161


In [ ]:
#Compute daily volatility
df['Volatility'] = df['High'] - df['Low']
df.head()

,Date,Close,High,Low,Open,Volume,Daily_Return,Volatility
0,2020-01-02,1524.500000,1528.699951,1518.000000,1518.099976,214.0,NaN,10.699951
1,2020-01-03,1549.199951,1552.699951,1530.099976,1530.099976,107.0,0.016202,22.599976
2,2020-01-06,1566.199951,1580.000000,1560.400024,1580.000000,416.0,0.010973,19.599976
3,2020-01-07,1571.800049,1576.300049,1558.300049,1558.300049,47.0,0.003576,18.000000
4,2020-01-08,1557.400024,1604.199951,1552.300049,1579.699951,236.0,-0.009161,51.899902


In [ ]:
#Add new variable (Year) from the Date column
df['Year'] = df['Date'].dt.year
df.head()

,Date,Close,High,Low,Open,Volume,Daily_Return,Volatility,Year
0,2020-01-02,1524.500000,1528.699951,1518.000000,1518.099976,214.0,NaN,10.699951,2020
1,2020-01-03,1549.199951,1552.699951,1530.099976,1530.099976,107.0,0.016202,22.599976,2020
2,2020-01-06,1566.199951,1580.000000,1560.400024,1580.000000,416.0,0.010973,19.599976,2020
3,2020-01-07,1571.800049,1576.300049,1558.300049,1558.300049,47.0,0.003576,18.000000,2020
4,2020-01-08,1557.400024,1604.199951,1552.300049,1579.699951,236.0,-0.009161,51.899902,2020


In [ ]:
#Handling Missing Values
df.isnull().sum()
df.dropna(inplace=True)
df.head()

,Date,Close,High,Low,Open,Volume,Daily_Return,Volatility,Year
1,2020-01-03,1549.199951,1552.699951,1530.099976,1530.099976,107.0,0.016202,22.599976,2020
2,2020-01-06,1566.199951,1580.000000,1560.400024,1580.000000,416.0,0.010973,19.599976,2020
3,2020-01-07,1571.800049,1576.300049,1558.300049,1558.300049,47.0,0.003576,18.000000,2020
4,2020-01-08,1557.400024,1604.199951,1552.300049,1579.699951,236.0,-0.009161,51.899902,2020
5,2020-01-09,1551.699951,1555.699951,1543.300049,1555.699951,54.0,-0.003660,12.399902,2020


In [ ]:
#Outliers analyzing
Q1 = df['Daily_Return'].quantile(0.25)
Q3 = df['Daily_Return'].quantile(0.75)
IQR = Q3 - Q1

outliers = df[(df['Daily_Return'] < Q1 - 1.5*IQR) |
              (df['Daily_Return'] > Q3 + 1.5*IQR)]
print(outliers.head())

         Date        Close         High          Low         Open  Volume  \
39 2020-02-28  1564.099976  1642.500000  1564.099976  1640.300049   289.0   
41 2020-03-03  1642.099976  1645.300049  1594.000000  1594.500000   610.0   
48 2020-03-12  1589.300049  1647.500000  1566.400024  1637.000000   256.0   
49 2020-03-13  1515.699951  1589.800049  1513.800049  1574.900024    89.0   
51 2020-03-17  1524.900024  1537.699951  1469.300049  1469.300049   122.0   

    Daily_Return  Volatility  Year  
39     -0.046281   78.400024  2020  
41      0.031275   51.300049  2020  
48     -0.031741   81.099976  2020  
49     -0.046310   76.000000  2020  
51      0.026247   68.399902  2020  


Since the outliers is important in analyze the effects of geopolitical events we decide to kept it.

In [ ]:
#Saved cleaned and processed dataset
df.to_csv("gold_prices_cleaned.csv", index=False)


# EXPLORATORY DATA ANALYSIS - GOLD PRICES & GEOPOLITICAL EVENTS


##1. INTRODUCTION

##Performing Exploratory Data Analysis (EDA) on:

### **Primary Data:**
#### **Gold price data from 2020-2026:**
##### - Price trends over time
##### - Daily returns distribution and patterns
##### - Volatility analysis
##### - Correlation between market metrics
##### - Identification of extreme events and anomalies

#### **Geopolitical news headlines from 2020-2026:**
##### - News frequency and patterns
##### - Event type distribution
##### - Text analysis of news content
##### - Timeline of geopolitical events

### **Integrated Analysis:** Combining both datasets
#### - Impact of events on gold prices
#### - Volatility changes around events
#### - Return patterns on event vs non-event days

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

#Determinig the styles and colors of the graphs
sns.set_theme(style="whitegrid")
vibrant_gold = "#FFD700"
sky_blue = "#1E90FF"
sunset_orange = "#FF4500"

# Loading the gold price data
df = pd.read_csv('gold_prices_cleaned.csv')
df['Date'] = pd.to_datetime(df['Date'])


#Checking how big our data is and what kind of information we have.
print("Checking the size of the dataset...")
print(f"Dataset Size: {df.shape[0]} rows and {df.shape[1]} columns")

# Looking at the data types (Advanced_eda style)
print("\nWhat kinds of data are in each column?")
print(pd.value_counts(df.dtypes))

# Visualizing how unique the data is
plt.figure(figsize=(10, 5))
unique_counts = df.nunique().sort_values()
unique_counts.plot.bar(color=sky_blue, title="Unique Values per Column (Log Scale)")
plt.yscale('log')
plt.ylabel("Unique Count")
plt.xlabel("Columns")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


#Checking for any missing information or messy data.
print("\nAre there any missing values in our dataset?")
print(df.isnull().sum())


#STATISTICAL SUMMARIES
print("\nSummary statistics for our gold price data:")
print(df.describe())

#TREND ANALYSIS (Visualizing Prices)
#Checking how gold prices changed over time
plt.figure(figsize=(14, 7))
plt.plot(df['Date'], df['Close'], color=vibrant_gold, linewidth=2, label='Closing Price')
plt.fill_between(df['Date'], df['Close'], color=vibrant_gold, alpha=0.1)
plt.title('Gold Price Trends: 2020 - 2026', fontsize=16, fontweight='bold')
plt.xlabel('Year', fontsize=12)
plt.ylabel('Price (USD)', fontsize=12)
plt.legend()
plt.show()

#PATTERN ANALYSIS (Distribution of Returns)
#This helps in determining the most common daily price changes
plt.figure(figsize=(12, 6))
sns.histplot(df['Daily_Return'], bins=60, kde=True, color=sunset_orange)
plt.title('Daily Returns Pattern (Normal Distribution?)', fontsize=16, fontweight='bold')
plt.xlabel('Daily Return Percentage')
plt.show()

#CORRELATION ANALYSIS (Feature Relationships)
plt.figure(figsize=(10, 8))
numeric_only = df.select_dtypes(include=[np.number])
correlation_matrix = numeric_only.corr()
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
sns.heatmap(correlation_matrix, mask=mask, annot=True, cmap='YlGnBu', fmt='.2f', square=True)
plt.title('Correlation Heatmap: How features connect', fontsize=16, fontweight='bold')
plt.show()

#ANOMALY DETECTION
#Spotting price jumps (outliers)
plt.figure(figsize=(12, 6))
sns.boxplot(x='Year', y='Daily_Return', data=df, palette="husl")
plt.title('Yearly Daily Returns: Spotting Anomalies', fontsize=16, fontweight='bold')
plt.xlabel('Year')
plt.ylabel('Daily Return')
plt.show()

###Final EDA Summary (Gold Prices Primary Data)

1. Price Trends and Patterns

 Gold prices were steady for a few years, but starting in 2024, they took off, . Looking at the “daily returns” (how much it changes each day), most of the movements are small and centered around 0%, following a standard bell curve. This means the price growth is consistent rather than just random jumps.

2. Anomalies (The “Weird” Days)

By using boxplots for each year, we spotted some clear outliers. 2020 and 2022 had some crashes, where the price dropped much harder than usual. However, 2024 and 2025 looks a lot more stable even though the price is higher. This suggests the current bull market has less extreme panic than we saw a few years ago.

3. Key Relationship (Correlation)
The heatmap showed that while all the price points (Open, Close, High, Low) move perfectly together, Volume actually has almost no relationship with the price.This is an important insight because it tells us that a high number of trades does not necessarily mean the price will move in a certain direction.

At the end commenting on what was mentioned above, the data is high quality and shows a strong, trending market with occasional historical shocks.